In [2]:
# 轻量级数据库
import sqlite3


In [4]:
# 打开链接
conn = sqlite3.connect("test2.db")
# 数据库操作的句柄
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS  employees (
    id INTEGER PRIMARY KEY,
    name TEXT,
    department TEXT,
    salary INTEGER
)
""")

In [6]:
sample_data = [
    (6, "陈昊", "开发部", 32000),
    ( 7, "张三", "销售部", 20000),
    ( 8, "曹威威", "开发部", 33000),
    ( 9, "李四", "销售部", 15000)
]
cursor.executemany('INSERT INTO employees VALUES (?, ?, ?, ?)', sample_data)
conn.commit()

In [7]:
from openai import OpenAI
client = OpenAI(
    api_key='sk-6df98a8a02cf42b4b69e5175de1d57a0',
    base_url='https://api.deepseek.com/v1'
)

In [8]:
# 通过table_info 拿到employees 表的描述
# llm 生成sql 提供上下文
schema = cursor.execute("PRAGMA table_info(employees)").fetchall()
schema_str = "CREATE TABLE EMPLOYEES (\n" + "\n".join([f"{col[1]} {col[2]}" for col in schema]) + "\n)"
print("数据库Schema:")
print(schema_str)

数据库Schema:
CREATE TABLE EMPLOYEES (
id INTEGER
name TEXT
department TEXT
salary INTEGER
)


In [11]:
def ask_deepseek(query, schema):
    prompt = f"""
    这是一个数据库的Schema:
    {schema}
    根据这个Schema,你能输出一个SQL查询来回答以下问题吗？
    只输出SQL查询,不要输出任何其他内容，也不要带任何格式。
    问题：{query}
    """
    # print(prompt)
    response = client.chat.completions.create(
        model="deepseek-reasoner",
        max_tokens=2048,
        messages=[{
            "role": "user",
            "content": prompt
        }]
    )
    return response.choices[0].message
    
# ask_deepseek("开发部部门员工的姓名和工资是多少？", schema)
question = "开发部部门员工的姓名和工资是多少？"
sql = ask_deepseek(question, schema_str)
print("生成的sql查询：")
print(sql)


生成的sql查询：
ChatCompletionMessage(content="```sql\nSELECT name, salary FROM EMPLOYEES WHERE department = '开发部';\n```", refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, reasoning_content="我们只需要查询开发部（department为'开发部'）的员工姓名和工资。\n 根据表结构，我们可以选择name和salary字段，并添加条件department = '开发部'。")
